In [1]:
import pandas as pd
import ast

In [2]:
data_folder = '/home/marta/jku/2025_ws_mmsr/data/'
full_data_folder = f'{data_folder}/full'
retrieval_data_folder = f'{data_folder}/retrieval'
binary_classification_data_folder = f'{data_folder}/binary_classification'

In [3]:
track_genres = pd.read_csv(f'{full_data_folder}/id_genres_mmsr.tsv', sep='\t')

In [4]:
# we select 1000 randomly sampled tracks to create the pairs
track_genres = track_genres.sample(n=1000, random_state=42).reset_index(drop=True)

In [5]:
track_genres['genre'] = track_genres['genre'].apply(lambda x: ast.literal_eval(str(x)))

In [6]:
samples = set([(el1, el2) for el1 in track_genres.index for el2 in track_genres.index]) # if el1 < el2]) allow for A-B as B-A

In [7]:
len(samples)

1000000

In [8]:
i_s = [i for i, j in samples]
j_s = [j for i, j in samples]

In [9]:
i_track_genres = track_genres.loc[i_s]
i_track_genres.columns = ['i', 'i_genres']
i_track_genres = i_track_genres.reset_index(drop=True)
j_track_genres = track_genres.loc[j_s]
j_track_genres.columns = ['j', 'j_genres']
j_track_genres = j_track_genres.reset_index(drop=True)

In [10]:
i_track_genres.head(1)

,i,i_genres
0,9XQknh6aLMYeJlpH,"[pop, rock, pop rock, teen pop, disney, funk r..."


In [11]:
dataset = i_track_genres.merge(j_track_genres, left_index=True, right_index=True)[['i', 'j', 'i_genres', 'j_genres']]

In [12]:
dataset.head(1)

,i,j,i_genres,j_genres
0,9XQknh6aLMYeJlpH,bTZAQWb0rLzftxrJ,"[pop, rock, pop rock, teen pop, disney, funk r...","[symphonic metal, power metal, symphonic power..."


In [13]:
dataset['is_match'] = dataset.apply(lambda row: len(set(row['i_genres']).intersection(set(row['j_genres']))) > 0, axis=1)

In [14]:
dataset = dataset[['i', 'j', 'is_match', 'i_genres', 'j_genres']]

In [15]:
dataset.head(3)

,i,j,is_match,i_genres,j_genres
0,9XQknh6aLMYeJlpH,bTZAQWb0rLzftxrJ,False,"[pop, rock, pop rock, teen pop, disney, funk r...","[symphonic metal, power metal, symphonic power..."
1,VFllAQWdcfpTHYOL,LQstYWe4jn4FqaNN,True,"[pop, jazz, soft rock, rock, singer songwriter...",[rock]
2,e0xraBYvAOVP3hlY,C8hHrdzeQLfPqvL8,False,"[folk, singer songwriter, lo fi, poetry, indie...",[disco]


In [16]:
# shuffle
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
# select train val test
n = len(dataset)
train = dataset.iloc[:int(0.8*n)]
val = dataset.iloc[int(0.8*n):int(0.9*n)]
test = dataset.iloc[int(0.9*n):]

In [17]:
train.to_csv(f'{binary_classification_data_folder}/binary_train.tsv', sep='\t', index=False)
val.to_csv(f'{binary_classification_data_folder}/binary_val.tsv', sep='\t', index=False)
test.to_csv(f'{binary_classification_data_folder}/binary_test.tsv', sep='\t', index=False)
dataset.to_csv(f'{binary_classification_data_folder}/binary_full.tsv', sep='\t', index=False)

In [18]:
binary_classification_data_folder

'/home/marta/jku/2025_ws_mmsr/data//binary_classification'